# LS-NEQ 13_prereg_evaluation — Präregistrierte Auswertung der vorhandenen Läufe

**Auswertung über die Stufen C, D, G–L** (`lsne.prereg_eval`, siehe `docs/PREREGISTRATION.md` §2–4 und
`docs/STAGES.md`). Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Die konfirmatorischen Endpunkte aus den Laufverzeichnissen berechnen, die schon auf Drive liegen —
ohne eine GPU-Stufe neu zu rechnen. Bestätigungsmenge: Testfamilien F01–F12 × Formulierungen {A, B}
(Entwicklungsfamilien DEV01–DEV04 bleiben außen vor); Statistik: Mittel der Familienmittel mit
Familien-Cluster-Bootstrap (10 000 Ziehungen, beide Formulierungen einer Familie gemeinsam gezogen);
Benjamini-Hochberg über die Sekundärgruppe 1–6, 8–9 (wenn G0 besteht), 10, 13 und 14 (8, 9, 13, 14 mit dem
`p_boot` ihrer Tabelle; Tabellen ohne `p_boot` bleiben unkorrigiert und höchstens „offen“). Mindesteffekte
entscheidet das Intervall (untere Grenze ≥ Mindesteffekt), nie der Punktschätzer.

**Laufzeit.** Eine **CPU-Laufzeit genügt** (Laufzeit → Laufzeittyp ändern → CPU): es wird kein Modell geladen.
Die Auswertung liest nur; der Checkpoint-Sweep (Notebook 05, Stufe F) wird nie geöffnet, ein noch laufender
Sweep wird also nicht gestört.

**Eingaben.** Die Laufwurzeln `<wurzel>/<stufe>/<modell>@<revision>/` mit `manifest.json` (`status: ok`):
standardmäßig `MyDrive/LSNE/runs` (+ `runs_trace`) und, falls vorhanden, `MyDrive/LSNE_B/runs` (+ `runs_trace`),
die Verknüpfung zum Ordner des zweiten Kontos (siehe nächste Zelle). Liegt eine Stufe unter mehreren Wurzeln,
gilt der neueste Lauf; Stufe L wird in den Stand-in-Lauf und den Spurenmodell-Lauf getrennt.

**Ausgaben.** `prereg_results.csv` (eine Zeile je Endpunkt und Unterzeile: `id, name, estimate, ci_low, ci_high,
p_two_sided, p_one_sided, n_families, n_rows, rule, verdict, bh_q, deviation_note` + `role, stage, direction,
bh_group, bh_q_endpoint, source, min_effect`), `prereg_results.md` (Bericht), `prereg_runs.csv` (gefundene Läufe) und
`prereg_meta.json` unter `<OUT_ROOT>/prereg_eval/<modell>@<revision>/`; eine Abbildung unter
`<OUT_ROOT>/figures/13_prereg_evaluation/`; am Ende eine kurze `summary_13_prereg_evaluation_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Wurzeln wählen → Auswertung → gefundene Läufe → Ergebnistabelle und Bericht →
Übersichtsabbildung → Zusammenfassung.

**Präregistrierung.** Primärer Endpunkt (§2): Δλ(K) = λ(K>0, `lang_dom`) − λ(K=0, `lang_dom`) aus Stufe D
(`damage`, Tiefen-Injektion), nur Zelle C1 (die einzige, deren Injektionsposition die präregistrierte ist:
letztes Token von Einheit 9), dieselbe Differenz für `rand` (Mittel über die Zufallsrichtungen) und die
Differenz der Differenzen; gestützt nur, wenn die untere KI-Grenze von Δλ(K=8) den Mindesteffekt 0,05
erreicht (dieselbe Intervallregel wie `damage_boot` in Stufe D), das KI der Differenz der Differenzen positiv
ist und die Punktschätzer monoton in K steigen; „offen“, wenn das Intervall den Mindesteffekt weder sichert
noch ausschließt. Ohne eine auf DEV eingefrorene Dosis werden alle präregistrierten Dosen berichtet.
C2/C3/seg erscheinen nur als Abweichungszeilen. §4: Läufe mit `audit_rel_err` > 1e-4 sind ungültig (Zeilen
`audit.D1`, `audit.C`). Kontrast 6: `ent` im Vorzeichen von State 2143 gegen Zufall, einseitig, über alle
Standardvarianten gepoolt; das umgekehrte Vorzeichen ist eine Kontrolle. Kontrast 7 (G0) ist ein Gate: verfehlt
es, werden 8–9 nicht interpretiert. 10 (ii)/(iii) werden auf den Testfamilien neu berechnet (konstante
zensierte Reihen zählen als ρ = 0, die Variante „wie gelaufen“ ist eine Sensitivitätszeile); 10f liest die auf
DEV eingefrorenen Merkmale (`lambda_features_frozen`). 11–18 werden aus ihren Tabellen übernommen (ein
`trace_verdict` nach der alten Regel wird neu beurteilt, nie übernommen); die bekannten Einschränkungen stehen
in `deviation_note`.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Abschnitt druckt den Traceback und das
Notebook läuft weiter). Fehlt eine Stufe, erscheint der Endpunkt mit `not_tested` und dem Grund.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


## Zweites Konto einbinden (`LSNE_B`)

Die Läufe liegen auf zwei Google-Konten. Damit dieses Notebook beide sieht, wird der Ordner des anderen
Kontos als **Verknüpfung** in die eigene Ablage gelegt:

1. Im anderen Konto den Ordner `LSNE` für dieses Konto freigeben (Lesezugriff genügt).
2. In diesem Konto in Google Drive „Für mich freigegeben“ öffnen, Rechtsklick auf `LSNE` →
   „Organisieren“ → „Verknüpfung hinzufügen“ → „Meine Ablage“.
3. Die Verknüpfung in **`LSNE_B`** umbenennen (eine Verknüpfung namens `LSNE` würde mit dem eigenen
   Ordner kollidieren).

Nach dem Einbinden erscheint sie als `/content/drive/MyDrive/LSNE_B`; die Konfigurationszelle nimmt
`LSNE_B/runs` (und `LSNE_B/runs_trace`) dann automatisch dazu. Andere Wurzeln lassen sich im Feld
`RUN_ROOTS_TEXT` oder in der Eingabeaufforderung komma-getrennt angeben.


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration: Laufwurzeln und Auswertungsparameter -------------------------------
NOTEBOOK = "13_prereg_evaluation"
STAGE = "A-L"
MODEL_TAG = "pythia-160m"          # Modell-Tag der Laufverzeichnisse (<stufe>/<modell>@<revision>/)
REVISION = "step98000"             # step98000 (Bestätigungslauf); step99000 für die 99k-Hälfte von Kontrast 11
N_BOOT = 10000                     # präregistriert: 10 000 Familien-Bootstrap-Ziehungen
SEED = 0
FROZEN_DOSE = None                 # auf DEV01-DEV04 eingefrorene Schadensdosis (z. B. 0.05); None = alle Dosen berichten
RUN_ROOTS_TEXT = ""  # @param {type:"string"}
ASK = True                         # True: nach den Wurzeln fragen, wenn RUN_ROOTS_TEXT leer ist (Enter = Standard)

MYDRIVE = "/content/drive/MyDrive"
default_roots = [os.path.join(MYDRIVE, base, child) for base in ("LSNE", "LSNE_B") for child in ("runs", "runs_trace")
                 if os.path.isdir(os.path.join(MYDRIVE, base, child))]
if not default_roots:              # outside Colab: the local runs root of lsne.io.drive_root()
    default_roots = [RUNS_ROOT]
if ASK and not RUN_ROOTS_TEXT.strip():
    try:
        RUN_ROOTS_TEXT = input(f"Laufwurzeln, komma-getrennt (Enter = {', '.join(default_roots)}): ")
    except Exception:              # no stdin (headless execution): use the defaults
        RUN_ROOTS_TEXT = ""
RUN_ROOTS = [r.strip() for r in RUN_ROOTS_TEXT.split(",") if r.strip()] or default_roots
if os.path.isdir(MYDRIVE) and not os.path.isdir(os.path.join(MYDRIVE, "LSNE_B")):
    print("Hinweis: kein MyDrive/LSNE_B -> die Läufe des zweiten Kontos fehlen (Verknüpfung siehe Markdown oben).")
EVAL_OUT = os.path.join(OUT_ROOT, "prereg_eval", f"{MODEL_TAG}@{REVISION}")
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)
CLI_COMMAND = ("python -m lsne.prereg_eval " + " ".join(f"--runs {r}" for r in RUN_ROOTS)
               + f" --out {EVAL_OUT} --model {MODEL_TAG} --revision {REVISION} --n-boot {N_BOOT} --seed {SEED}"
               + (f" --frozen-dose {FROZEN_DOSE}" if FROZEN_DOSE is not None else ""))
print("Laufwurzeln:")
for r in RUN_ROOTS:
    print(f"  {r}  ({'vorhanden' if os.path.isdir(r) else 'FEHLT'})")
print("Ausgabe:", EVAL_OUT)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Hilfsfunktionen: Anzeige und abgesicherte Abschnitte --------------------------------
import contextlib
import json
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt


def show(df, n: int = 60):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Auswertung: alle präregistrierten Endpunkte aus den vorhandenen Tabellen -----------
# CLI-Äquivalent: python -m lsne.prereg_eval --runs <wurzel> [--runs <wurzel2>] --out <dir> --n-boot 10000
from lsne import prereg_eval as PE

res = PE.evaluate(RUN_ROOTS, model=MODEL_TAG, revision=REVISION, n_boot=N_BOOT, seed=SEED, frozen_dose=FROZEN_DOSE)
paths = PE.write_outputs(res, EVAL_OUT)
for k, p in paths.items():
    print(f"{k:9s} {p}")
for n in res["meta"]["notes"]:
    print("Hinweis:", n)


In [ ]:
# --- Gefundene Läufe: welcher Lauf je Stufe verwendet wird ---------------------------
with guarded("Laufverzeichnisse"):
    runs = res["runs"]
    show(runs[["key", "used", "status", "timestamp", "n_boot", "path", "note"]], n=len(runs))
    used = set(runs.loc[runs["used"].astype(bool), "key"])
    missing = [k for k in PE.EVAL_KEYS if k not in used]
    if missing:
        print("Keine verwendbaren Läufe für:", ", ".join(missing), "-> diese Endpunkte erscheinen als not_tested.")


In [ ]:
# --- Ergebnistabelle (alle Endpunkte) und Bericht --------------------------------------
with guarded("Ergebnistabelle"):
    tab = res["results"]
    print("Urteile:", tab["verdict"].value_counts().to_dict())
    print("BH-Gruppe:", res["bh"]["m"], "Tests; Endpunkt-Gruppierung (Sensitivität):", res["bh"]["m_endpoints"], "Endpunkte")
    cols = ["id", "name", "estimate", "ci_low", "ci_high", "p_two_sided", "p_one_sided", "bh_q", "n_families",
            "n_rows", "verdict", "deviation_note"]
    main = tab[tab["role"].isin(["primary", "audit", "secondary", "gate"])]
    show(main[cols], n=len(main))
    print("Weitere Zeilen (Komponenten, Abweichungen, bedingt, übernommen, Sensitivität):",
          int(len(tab) - len(main)), "-> siehe", paths["results"])
with guarded("Bericht (prereg_results.md)"):
    text = paths["markdown"].read_text(encoding="utf-8")
    try:
        from IPython.display import Markdown, display
        display(Markdown(text))
    except ImportError:
        print(text[:6000])


In [ ]:
# --- Übersicht: Schätzer mit 95 %-KI des Primär- und der Sekundärendpunkte ----------------
with guarded("Übersichtsabbildung"):
    tab = res["results"]
    sel = tab[tab["role"].isin(["primary", "secondary"]) & np.isfinite(tab["ci_low"].astype(float))]
    if sel.empty:
        print("Keine Zeile mit Konfidenzintervall (fehlende Stufen?).")
    else:
        colors = {"supported": "C2", "not_supported": "C3", "open": "C0", "not_interpretable": "grey"}
        fig, ax = plt.subplots(figsize=(8, 0.28 * len(sel) + 1.5))
        y = np.arange(len(sel))[::-1]
        for yi, (_, r) in zip(y, sel.iterrows()):
            c = colors.get(r["verdict"], "k")
            ax.plot([r["ci_low"], r["ci_high"]], [yi, yi], color=c, lw=2)
            ax.plot([r["estimate"]], [yi], "o", color=c, ms=4)
        ax.axvline(0, color="k", lw=0.6)
        ax.set_yticks(y)
        ax.set_yticklabels(sel["id"], fontsize=7)
        ax.set_xlabel("Schätzer [95 %-KI] (Einheit des Endpunkts)")
        ax.set_title("Präregistrierte Endpunkte: grün gestützt, rot nicht gestützt, blau offen", fontsize=9)
        path = os.path.join(FIG_DIR, f"endpoints_{MODEL_TAG}_{REVISION}.png")
        fig.savefig(path, dpi=120, bbox_inches="tight")
        print("Abbildung:", path)
        plt.show()


In [ ]:
# --- Kurze Zusammenfassung nach OUT_ROOT schreiben -------------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    tab = res["results"]
    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=MODEL_TAG, revision=REVISION, n_boot=N_BOOT, seed=SEED,
        frozen_dose=FROZEN_DOSE, run_roots=RUN_ROOTS, out=EVAL_OUT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        runs_used={k: p for k, p, u in zip(res["runs"]["key"], res["runs"]["path"], res["runs"]["used"]) if u},
        verdicts=tab["verdict"].value_counts().to_dict(),
        primary=tab[tab["role"] == "primary"][["id", "estimate", "ci_low", "ci_high", "verdict"]].to_dict("records"),
        bh=dict(m=res["bh"]["m"], m_endpoints=res["bh"]["m_endpoints"]),
        files={k: str(p) for k, p in paths.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{MODEL_TAG}_{REVISION}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=1, ensure_ascii=False, default=str)
    print("Zusammenfassung:", path)
    print(json.dumps(summary, indent=1, ensure_ascii=False, default=str)[:3000])
